# BT5151 Credit Risk Monitoring Pipeline

This notebook runs the real LangGraph pipeline for the BT5151 credit risk project. There is no manual fallback path in this notebook: preprocessing, training, evaluation, model selection, inference, business explanation, and action recommendation all flow through the compiled graph.

Before running the graph, add your `OPENAI_API_KEY` to `.env` in the project root.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn gradio langgraph pydantic python-dotenv openai openpyxl


In [ ]:
from pathlib import Path
import os
import sys

import gradio as gr
import pandas as pd
from dotenv import load_dotenv

sys.path.append(str(Path.cwd() / "src"))
load_dotenv()

from bt5151_credit_risk.graph import compile_graph
from bt5151_credit_risk.profile import build_dataset_profile


In [ ]:
DATA_PATH = Path("train.csv")
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
dataset_profile = build_dataset_profile(raw_df)
dataset_profile["row_count"], dataset_profile["target_distribution"]


In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Put it in .env before running the graph.")

graph = compile_graph()
graph


In [ ]:
def run_pipeline(row_index: int):
    return graph.invoke(
        {
            "raw_dataset_path": str(DATA_PATH),
            "inference_input": {"row_index": int(row_index)},
        }
    )


sample_result = run_pipeline(0)
sample_result["selected_model_name"], sample_result["prediction_output"], sample_result["risk_explanation"], sample_result["recommended_action"]


In [ ]:
def predict_from_row(row_index):
    result = run_pipeline(row_index)
    prediction = result["prediction_output"]
    explanation = result["risk_explanation"]
    action = result["recommended_action"]
    return (
        prediction["predicted_label"],
        explanation["summary"],
        action["action"],
        action["reason"],
        result["selected_model_name"],
        prediction["confidence"],
    )


demo = gr.Interface(
    fn=predict_from_row,
    inputs=gr.Slider(minimum=0, maximum=len(raw_df) - 1, step=1, value=0, label="Customer Row Index"),
    outputs=[
        gr.Textbox(label="Predicted Credit Score"),
        gr.Textbox(label="Risk Summary"),
        gr.Textbox(label="Recommended Action"),
        gr.Textbox(label="Action Reason"),
        gr.Textbox(label="Selected Model"),
        gr.Number(label="Top Confidence"),
    ],
    title="BT5151 Credit Risk Monitor",
    description="Run the full LangGraph credit risk pipeline for a selected row from the training dataset.",
)

RUN_GRADIO = False
if RUN_GRADIO:
    demo.launch(share=True)
